# Bài 04: Vấn đề Mất cân bằng Dữ liệu (Imbalanced Data)

**Mục tiêu bài học:**
- Định nghĩa chính xác dữ liệu mất cân bằng là gì.
- Nhận biết các ví dụ thực tế phổ biến của bài toán này.
- Hiểu tại sao các mô hình Machine Learning truyền thống thường hoạt động kém trên dữ liệu mất cân bằng.
- Phân tích tác động của dữ liệu mất cân bằng lên các chỉ số đánh giá đã học.

## Dữ liệu mất cân bằng là gì?

**Dữ liệu mất cân bằng (Imbalanced Data)** là một thuật ngữ trong bài toán phân loại, mô tả tình huống mà số lượng các mẫu trong các lớp (categories) không đồng đều một cách đáng kể.

Thông thường, chúng ta sẽ có:
- **Lớp đa số (Majority Class):** Lớp chiếm phần lớn số lượng mẫu. Thường được gán nhãn là `0` hoặc `Negative`.
- **Lớp thiểu số (Minority Class):** Lớp chiếm phần rất nhỏ số lượng mẫu. Đây thường là lớp mà chúng ta quan tâm đến việc phát hiện. Thường được gán nhãn là `1` hoặc `Positive`.

**Mức độ mất cân bằng có thể khác nhau:**
- **Mất cân bằng nhẹ:** Tỷ lệ 80/20
- **Mất cân bằng vừa:** Tỷ lệ 90/10 hoặc 95/5
- **Mất cân bằng nghiêm trọng:** Tỷ lệ 99/1 hoặc 99.9/0.1

Đây là một vấn đề cực kỳ phổ biến trong các bài toán thực tế, không phải là một trường hợp ngoại lệ.

### Ví dụ thực tế

1.  **Phát hiện gian lận thẻ tín dụng:**
    - Lớp đa số: Các giao dịch hợp lệ (99.9%+).
    - Lớp thiểu số: Các giao dịch gian lận (rất hiếm).

2.  **Chẩn đoán y khoa:**
    - Lớp đa số: Bệnh nhân không mắc bệnh hiếm nào đó.
    - Lớp thiểu số: Bệnh nhân mắc bệnh hiếm.

3.  **Dự đoán khách hàng rời bỏ (Customer Churn):**
    - Lớp đa số: Khách hàng tiếp tục sử dụng dịch vụ.
    - Lớp thiểu số: Khách hàng hủy dịch vụ.

4.  **Kiểm soát chất lượng sản xuất:**
    - Lớp đa số: Các sản phẩm đạt chuẩn.
    - Lớp thiểu số: Các sản phẩm bị lỗi.

5.  **Tỷ lệ click quảng cáo (Ad Click-Through Rate):**
    - Lớp đa số: Người dùng thấy quảng cáo nhưng không click.
    - Lớp thiểu số: Người dùng click vào quảng cáo.

### Trực quan hóa dữ liệu mất cân bằng

Hãy tạo một tập dữ liệu giả lập để thấy rõ sự chênh lệch.

In [ ]:
from sklearn.datasets import make_classification
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Tạo dữ liệu với tỷ lệ 95/5
X, y = make_classification(n_samples=1000, n_features=2, n_informative=2, 
                           n_redundant=0, n_repeated=0, n_classes=2, 
                           n_clusters_per_class=1, weights=[0.95, 0.05], 
                           flip_y=0, random_state=42)

df = pd.DataFrame(X, columns=['feature_1', 'feature_2'])
df['target'] = y

# Đếm số lượng mỗi lớp
print("Số lượng mẫu mỗi lớp:")
print(df['target'].value_counts())

# Vẽ biểu đồ countplot
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df)
plt.title('Phân phối các lớp (Tỷ lệ 95/5)')
plt.show()

# Vẽ biểu đồ scatter plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x='feature_1', y='feature_2', hue='target', data=df, style='target')
plt.title('Phân bố dữ liệu trên không gian 2 chiều')
plt.show()

Biểu đồ cho thấy rõ sự áp đảo của lớp `0` (đa số) so với lớp `1` (thiểu số).

## Tại sao mô hình lại hoạt động kém?

Hầu hết các thuật toán Machine Learning được thiết kế với giả định rằng dữ liệu được phân phối cân bằng. Mục tiêu của chúng trong quá trình huấn luyện là **tối thiểu hóa tổng lỗi** trên toàn bộ tập huấn luyện.

Khi dữ liệu mất cân bằng, điều này gây ra một vấn đề lớn:

> Mô hình có thể đạt được tổng lỗi thấp bằng cách **tập trung học và dự đoán đúng lớp đa số**, trong khi **phớt lờ hoặc phân loại sai hoàn toàn lớp thiểu số**.

Hãy tưởng tượng một mô hình "lười biếng". Trong tập dữ liệu 95/5 ở trên, nếu nó luôn dự đoán lớp `0`, nó sẽ đúng 950/1000 lần. Accuracy của nó là 95%! Đối với hàm mất mát (loss function) của mô hình, đây là một kết quả rất tốt. Mô hình không có nhiều "động lực" để cố gắng học các đặc điểm của 50 mẫu lớp thiểu số ít ỏi, vì việc phân loại sai chúng chỉ làm tăng tổng lỗi lên một chút.

### Thí nghiệm: Huấn luyện mô hình trên dữ liệu mất cân bằng

Hãy xem một mô hình Logistic Regression hoạt động như thế nào trên tập dữ liệu chúng ta vừa tạo.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Huấn luyện mô hình
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Dự đoán và đánh giá
y_pred = model.predict(X_test)

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**Phân tích kết quả:**

- **Accuracy tổng thể (macro avg / weighted avg)** có vẻ rất cao (khoảng 95%). Đây chính là cái bẫy của Accuracy mà chúng ta đã nói.
- Nhìn vào lớp `1` (lớp thiểu số):
  - **Precision** rất cao (ở ví dụ này có thể là 1.00), nghĩa là khi mô hình dự đoán `1`, nó rất chắc chắn. Nhưng...
  - **Recall** lại rất thấp (ví dụ này là 0.20). Điều này có nghĩa là mô hình chỉ tìm thấy được **20%** trong tổng số các trường hợp `Positive` thật sự. **80%** các trường hợp quan trọng đã bị bỏ lỡ!
  - **F1-score** của lớp `1` cũng vì thế mà rất thấp.

Đây chính là triệu chứng kinh điển của một mô hình được huấn luyện trên dữ liệu mất cân bằng: nó có **Recall rất thấp đối với lớp thiểu số**.

## Tác động lên các chỉ số đánh giá

Hãy tóm tắt lại cách dữ liệu mất cân bằng "đánh lừa" các chỉ số của chúng ta:

1.  **Accuracy:** Trở nên vô nghĩa và nguy hiểm. Nó cho một cảm giác an toàn giả tạo vì giá trị của nó bị chi phối hoàn toàn bởi hiệu năng trên lớp đa số.

2.  **Precision, Recall, F1-Score:** Vẫn hữu ích, nhưng **bắt buộc phải xem xét cho từng lớp riêng biệt**, đặc biệt là lớp thiểu số. `classification_report` là công cụ tuyệt vời cho việc này. Đừng chỉ nhìn vào các giá trị trung bình (macro/weighted avg).

3.  **ROC AUC:** Vẫn có thể cho giá trị cao một cách đáng ngạc nhiên. Như đã thảo luận ở bài trước, vì FPR = FP / (FP + TN), và mẫu số `(FP + TN)` (tổng số lớp Negative) là rất lớn, nên để FPR tăng lên là rất khó. Đường cong ROC có thể trông rất tốt dù Precision trên lớp thiểu số rất tệ. Do đó, AUC có thể không phản ánh hết vấn đề.

4.  **Precision-Recall Curve:** Đây là công cụ chẩn đoán đáng tin cậy nhất trong bối cảnh này. Nó tập trung vào mối quan hệ giữa Precision và Recall của lớp Positive (thiểu số), cho thấy rõ sự sụt giảm của Precision khi mô hình cố gắng tăng Recall.

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

# Lấy điểm số xác suất
y_scores = model.predict_proba(X_test)[:, 1]

# Tính ROC AUC
roc_auc = roc_auc_score(y_test, y_scores)
print(f"ROC AUC Score: {roc_auc:.4f}")

# Tính diện tích dưới đường cong Precision-Recall (PR AUC)
precision, recall, _ = precision_recall_curve(y_test, y_scores)
pr_auc = auc(recall, precision)
print(f"Precision-Recall AUC Score: {pr_auc:.4f}")

# So sánh hai đường cong
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_scores)
ax1.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
ax1.plot([0, 1], [0, 1], 'k--')
ax1.set_title('ROC Curve')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend()

# Precision-Recall Curve
ax2.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})')
ax2.set_title('Precision-Recall Curve')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend()

plt.show()

Kết quả trên cho thấy một kịch bản điển hình: ROC AUC có thể rất cao (ví dụ > 0.9), nhưng PR AUC lại thấp hơn đáng kể. Điều này xác nhận rằng P-R curve là một công cụ đánh giá nghiêm khắc và thực tế hơn cho dữ liệu mất cân bằng.

## Tóm tắt

- **Dữ liệu mất cân bằng** xảy ra khi các lớp phân loại có số lượng mẫu chênh lệch lớn, và đây là một vấn đề rất phổ biến.
- Các mô hình truyền thống có xu hướng **thiên vị lớp đa số** và **bỏ qua lớp thiểu số**, dẫn đến **Recall thấp** trên lớp thiểu số.
- **Accuracy** và **ROC AUC** có thể cho kết quả lạc quan giả tạo. Cần phải hết sức cẩn thận khi diễn giải chúng.
- Để đánh giá đúng hiệu năng, cần tập trung vào các chỉ số của lớp thiểu số (như trong `classification_report`) và sử dụng **Precision-Recall Curve**.

Bây giờ chúng ta đã hiểu rõ vấn đề, trong các bài học tiếp theo, chúng ta sẽ khám phá các giải pháp để xử lý nó, bắt đầu với các chỉ số đánh giá được thiết kế riêng cho dữ liệu mất cân bằng.